# CHT.M1 – Molecular Data Acquisition and Representation
### Notebook 04. Molecular Representation & Visualization

**Version 1.0 - June, 2026. Monterrey**

**Author:** [Flavio F. Contreras-Torres](https://orcid.org/0000-0003-2375-131X). Tecnológico de Monterrey.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/CheminformaticsTutorial/blob/main/modules/m1_data_acquisition/notebooks/04_molecular_representation.ipynb)


---


### Contents

This notebook concludes the **Molecular Representation** stage of our workflow by focusing on the visual interpretation of chemical structures. After retrieving molecular records from **PubChem** and **ChEMBL**, we now transform textual chemical representations into visual molecular objects that can be inspected directly in Python.

The notebook begins with **Step 0: Chemical Library Initialization**, where we load the structured dataset and establish the connection between chemical identifiers, molecular strings, and computational representations. We then move to **Step 1: Molecular Visualization Engine**, introducing the conversion of structural strings into 2D depictions suitable for direct inspection.

The final phase focuses on **Step 2: Batch Processing and Visual Inspection**, where molecules are rendered systematically in grid format to support qualitative structural exploration, recognize recurring motifs, and strengthen intuition about the chemical diversity of the dataset.


This notebook is designed to be completed in approximately **60–120 minutes**.


---

### How to work

1. **Open the notebook**: Click on the "Open in Colab" button or use the link provided to open the tutorial in **Google Colab**.
2. **Create your workspace**: Once open, go to **File > Save a copy in Drive**. This is vital! You must work on your own copy to ensure your progress is saved.
    - **Tip**: Look at the top-left corner. If you see "Copy of...", you are ready to go!
3. **Solve the exercises**: Complete the missing parts of the code where indicated. You can run each cell to test your progress (Shift + Enter).
4. When you finish:
    - **Rename** your file following the convention:
        - `YourID_NB4_M1.ipynb`
    - **Download the file**: Go to File > Download > Download .ipynb.
    - **Upload** the downloaded file to the **CANVAS assignment module**.

**Do not** modify the notebook structure or function signatures unless explicitly stated.


---


## Molecular Representation and Visualization

Chemical data become more interpretable when textual molecular formats are transformed into visible chemical structures. Although computational workflows often rely on encoded representations such as **SMILES**, meaningful chemical interpretation depends on reconnecting those strings to the underlying molecular architecture.

Two-dimensional molecular depictions provide a practical and intuitive way to inspect chemical structures. While they do not capture full conformational behavior, they allow the rapid recognition of **recurring scaffolds**, functional group patterns, and broad differences in molecular architecture. In this way, structural visualization helps maintain a direct connection between molecular records and chemical intuition.


---


### Step 0: Chemical Library Initialization

This block serves as the **Molecular Entry Point**. Before we can visualize or interact with the chemical structures, we must load the curated library into memory. This step is essential for transitioning from abstract data to tangible chemical entities.

This block initializes the graphical environment. In a chemoinformatics workflow, **Matplotlib** is the bridge between raw numerical data and visual scientific insight.

In [ ]:
# Step 0
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

csv_file = "pparg_chembl_ml_dataset.csv"
df = pd.read_csv(csv_file)

print(df.shape)
df.head()


### Step 1: Molecular Visualization Engine

This step moves beyond numbers and graphs to reveal the actual chemical structures of the molecules in your library, allowing for a direct "visual audit" of the chemical scaffolds. This step transforms abstract **SMILES** strings into 2D structural images. By connecting to the **PubChem PUG-REST API**, we can fetch high-quality depictions of our compounds without needing to install heavy local chemoinformatics libraries like RDKit.

* The script selects a random sample of $N=24$ molecules to ensure a representative view of the library without overwhelming the API or the notebook's memory.
* The `smiles_to_cid` function sends the SMILES string to PubChem. It uses `urllib.parse.quote` to ensure special characters (like `#` for triple bonds or / for stereochemistry) are correctly interpreted by the web server.
* Once the PubChem Identifier (CID) is found, `cid_to_png` fetches a 250px PNG image of the 2D structure.
* The code includes `time.sleep` and `try/except` logic. This is critical when working with APIs to avoid being blocked ("rate-limited") and to handle cases where a specific SMILES might not be indexed in PubChem.
* Using `matplotlib.pyplot.subplot`, the script organizes the successfully retrieved images into a clean $6 \times 4$ grid, labeling each structure with its ChEMBL ID and its `Activity_Class`.

Visualizing the molecules allows you to confirm that the SMILES represent realistic drug-like compounds. It helps identify "artifacts," such as extremely large molecules or inorganic salts that might have slipped through the filtering process.

This teaches students how to use **REST APIs** as a powerful tool in bioinformatics. It demonstrates that you don't always need to perform calculations locally if a trusted public resource like the NIH (PubChem) can provide the data.

In [ ]:
# Step 1
import pandas as pd
import numpy as np
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
from urllib.parse import quote
import time

# --- Config ---
SMILES_COL = "SMILES"
ID_COL = "molecule_chembl_id"
N = 24
PER_ROW = 6
RANDOM_SEED = 0

IMG_SIZE = 250   # px
TIMEOUT = 30
SLEEP = 0.1      # para no saturar PubChem (puedes subir a 0.2 si hay rate-limit)

assert "df" in globals(), "DataFrame 'df' no está definido. Primero carga tu CSV en df."
assert SMILES_COL in df.columns, f"Columna '{SMILES_COL}' no encontrada en df."

# --- Sample molecules ---
df_show = (
    df.dropna(subset=[SMILES_COL])
      .sample(n=min(N, len(df)), random_state=RANDOM_SEED)
      .copy()
      .reset_index(drop=True)
)

def smiles_to_cid(smiles: str, timeout=30):
    """
    Convierte SMILES -> CID (primer CID devuelto) usando PUG-REST.
    Usa URL-encoding para evitar errores con caracteres especiales.
    """
    smi = quote(smiles, safe="")  # encode TODO (incluye / y #)
    base_url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles"
    endpoint = "cids/JSON"
    url = f"{base_url}/{smi}/{endpoint}"
    r = requests.get(url, timeout=timeout)
    if r.status_code != 200:
        return None
    data = r.json()
    cids = data.get("IdentifierList", {}).get("CID", [])
    return int(cids[0]) if cids else None

def cid_to_png(cid: int, size=250, timeout=30):
    """
    Baja la imagen 2D PNG desde PubChem por CID.
    """
    base_url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid"
    image_size = f"{size}x{size}"
    url = f"{base_url}/{cid}/PNG?image_size={image_size}"
    r = requests.get(url, timeout=timeout)
    if r.status_code != 200:
        return None
    return Image.open(BytesIO(r.content)).convert("RGBA")

images, labels = [], []
failed = 0

for _, row in df_show.iterrows():
    smi_raw = str(row[SMILES_COL]).strip()
    if not smi_raw:
        continue

    cid = smiles_to_cid(smi_raw, timeout=TIMEOUT)
    if cid is None:
        failed += 1
        time.sleep(SLEEP)
        continue

    img = cid_to_png(cid, size=IMG_SIZE, timeout=TIMEOUT)
    if img is None:
        failed += 1
        time.sleep(SLEEP)
        continue

    lbl = str(row.get(ID_COL, f"CID:{cid}"))
    if "Activity_Class" in df_show.columns:
        lbl = f"{lbl}\n{row.get('Activity_Class','')}"
    else:
        lbl = f"{lbl}\nCID:{cid}"

    images.append(img)
    labels.append(lbl)

    time.sleep(SLEEP)

# --- Plot grid ---
n = len(images)
print(f"Images retrieved: {n}/{len(df_show)} | Failed: {failed}")

if n == 0:
    # Debug rápido: imprime 3 SMILES para revisar que hay contenido real
    print("Ejemplos de SMILES usados (primeros 3):")
    print(df_show[SMILES_COL].head(3).tolist())
else:
    ncols = PER_ROW
    nrows = int(np.ceil(n / ncols))
    plt.figure(figsize=(ncols * 3, nrows * 3))

    for i, (img, lbl) in enumerate(zip(images, labels), start=1):
        ax = plt.subplot(nrows, ncols, i)
        ax.imshow(img)
        ax.set_title(lbl, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    

### Step 2: Batch Processing

This block implements a **Batch Visualization Strategy**. Instead of looking at a random sample, this code systematically processes the entire dataset in manageable pages. This is a professional-grade approach to performing a Visual Quality Audit on a complete chemical library.

In medicinal chemistry, a "representative sample" is often not enough. This code allows the researcher to perform a **Structural Audit** of every single molecule in the library to identify rare scaffolds or unusual chemical features.

This step teaches the concept of Batching, a fundamental skill in bioinformatics and large-scale data science for handling thousands of entries without crashing the environment.


#### Technical Explanation:

1. Pagination Logic: The `for start in range(0, total, PER_PAGE)` loop slices the DataFrame into segments of 24 molecules. This prevents memory overflow and keeps the notebook organized.
2. API Integration Pipeline:
    - Translation: Converts SMILES strings into PubChem CIDs (Compound Identifiers).
    - Retrieval: Fetches the official 2D PNG coordinate-based render from the NCBI servers.
3. Rate Limiting (`time.sleep`): A deliberate pause of 0.05 seconds is included to respect PubChem's server traffic policies (PUG-REST), ensuring the script isn't flagged as a malicious bot.
4. Dynamic Grid Generation: The script calculates the required number of rows (`nrows`) for each batch and creates a new Matplotlib figure for every "page" of results, including a clear title (e.g., "Molecules 1–24").


In [ ]:
# Step 2 
## It might take a while to retrieve all images, so we won't run this by default.
import pandas as pd
import numpy as np
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
from urllib.parse import quote
import time

# --- Config ---
SMILES_COL = "SMILES"
ID_COL = "molecule_chembl_id"

PER_PAGE = 24      # how many molecules per figure
PER_ROW = 6
IMG_SIZE = 250
TIMEOUT = 30
SLEEP = 0.05

assert "df" in globals(), "DataFrame 'df' not defined."
assert SMILES_COL in df.columns, f"Column '{SMILES_COL}' not found."

df_show = df.dropna(subset=[SMILES_COL]).reset_index(drop=True)

def smiles_to_cid(smiles):
    smi = quote(smiles, safe="")
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smi}/cids/JSON"
    r = requests.get(url, timeout=TIMEOUT)
    if r.status_code != 200:
        return None
    data = r.json()
    cids = data.get("IdentifierList", {}).get("CID", [])
    return int(cids[0]) if cids else None

def cid_to_png(cid):
    base_url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid"
    image_size = f"{IMG_SIZE}x{IMG_SIZE}"
    url = f"{base_url}/{cid}/PNG?image_size={image_size}"
    r = requests.get(url, timeout=TIMEOUT)
    if r.status_code != 200:
        return None
    return Image.open(BytesIO(r.content)).convert("RGBA")

# --- Loop over dataset in batches ---
total = len(df_show)
print(f"Total molecules: {total}")

for start in range(0, total, PER_PAGE):
    end = min(start + PER_PAGE, total)
    batch = df_show.iloc[start:end]

    images = []
    labels = []

    for _, row in batch.iterrows():
        smi = str(row[SMILES_COL]).strip()
        cid = smiles_to_cid(smi)
        if cid is None:
            continue

        img = cid_to_png(cid)
        if img is None:
            continue

        lbl = str(row.get(ID_COL, f"CID:{cid}"))
        if "Activity_Class" in row:
            lbl = f"{lbl}\n{row.get('Activity_Class','')}"

        images.append(img)
        labels.append(lbl)

        time.sleep(SLEEP)

    # --- Plot this batch ---
    n = len(images)
    ncols = PER_ROW
    nrows = int(np.ceil(n / ncols))

    plt.figure(figsize=(ncols * 3, nrows * 3))
    for i, (img, lbl) in enumerate(zip(images, labels), start=1):
        ax = plt.subplot(nrows, ncols, i)
        ax.imshow(img)
        ax.set_title(lbl, fontsize=8)
        ax.axis("off")

    plt.suptitle(f"Molecules {start+1}–{end}", fontsize=14)
    plt.tight_layout()
    plt.show()



### Summary and Next Step

You have successfully completed the Molecular Representation and Visualization stage of the workflow.

**Current State:**  
We now have the ability to:

- Convert molecular strings such as SMILES into computational molecular objects
- Render 2D molecular structures directly in Python
- Inspect retrieved compounds visually through systematic molecular grids
- Recognize recurring structural patterns and broad differences in molecular architecture
- Connect molecular identifiers and encoded representations with their underlying chemical structures

**Next Step:**  
In the next module, we will move from molecular representation to data curation and structural exploration. There, we will begin evaluating dataset consistency, inspecting class composition and bioactivity distributions, and preparing the chemical library for downstream analysis.

---


### EXERCISES

Test your skills with these three challenges. Use the code cells below to write your solutions.

---

**Exercise 1: From SMILES to Structure**
Choose one molecule from your dataset and inspect its SMILES string.

* **Question 1:** Can you identify, directly from the SMILES, whether the molecule contains at least one ring, one oxygen atom, or one nitrogen atom?
* **Question 2:** After rendering the molecule in 2D, which of these features become easier to recognize visually than textually?


**Scientific Insight:** This exercise helps you connect a linear molecular encoding with its underlying chemical structure.

---

**Exercise 2: Visual Comparison of Molecular Architectures**
Render two different molecules from the dataset side by side.

* **Question 1:** Which molecule appears more structurally complex?
* **Question 2:** Do you observe clear differences in ring systems, branching, or the presence of polar functional groups?


**Scientific Insight:** Even before calculating descriptors, visual inspection can reveal broad differences in molecular architecture that are often hidden in tables.

---

**Exercise 3: Grid-Based Visual Inspection**
Run the batch visualization code and inspect the first molecular grid.

* **Question 1:** Do you notice any recurring structural motifs across several molecules?
* **Question 2:** Are most compounds visually similar, or does the dataset appear structurally diverse?


**Scientific Insight:** This exercise introduces qualitative structural inspection as a first step toward understanding chemical diversity.

---

**Exercise 4: Debugging the Visualization Pipeline**
In the batch visualization workflow, we use a short delay such as `time.sleep(0.05)` between API requests.

* **Question 1**: If you remove this line and the `requests.get` starts returning Error 429: Too Many Requests, what happened? How would you modify the SLEEP variable to be "safer" for the PubChem servers?
* **Question 2:** Why is it important to make visualization pipelines respectful of server limits?


**Scientific Insight:** Good cheminformatics workflows are not only scientifically meaningful, but also technically robust and reproducible.

---

**Exercise 5: Representation vs. Interpretation**
Select one rendered molecule and examine both its SMILES string and its 2D depiction.

* **Question 1:** Which representation is more useful for computation?
* **Question 2:** Which representation is more useful for human interpretation?
* **Question 3:** Why do cheminformatics workflows need both encoded and visual representations?


**Scientific Insight:** Chemical data become more meaningful when we can move fluently between machine-readable representations and human-readable structural interpretation.

---


### Outlook

You have completed the **Molecular Representation and Visualization phase** of the workflow. By moving beyond textual and tabular molecular records, you can now translate encoded chemical information into visual molecular structures that are directly interpretable.

From here, you now have the ability to:

* Convert SMILES strings into computational molecular objects
* Render 2D depictions of chemical structures directly in Python
* Inspect compounds visually through systematic molecular grids
* Recognize recurring structural motifs and broad differences in molecular architecture
* Connect molecular identifiers and encoded records with their underlying chemical structures


See you in the next lesson!

---


### License  
The content of this tutorial itself is licensed under the terms and conditions of the [Creative Commons Attribution (CC BY 4.0) license](https://creativecommons.org/licenses/by/4.0/legalcode.en), and the underlying source code used to format and display that content is licensed under the [MIT license](https://github.com/NanoBiostructuresRG/NumpyTutorial/blob/main/LICENSE). See the LICENSE files for full details.

### Attribution
If you use or adapt this material, please provide appropriate credit to the original [authors](https://orcid.org/my-orcid?orcid=0000-0003-2375-131X) and repository:
[https://github.com/NanoBiostructuresRG](https://github.com/NanoBiostructuresRG)
